# Setup Inicial

## Importando Bibliotecas

In [2]:
import os
from google.cloud import bigquery

## Autenticando o Client do BigQuery

In [3]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../bigquery_api_key.json'
client = bigquery.Client()

# Questão 1 — Extração e Análise Exploratória com SQL

## 1.1 — Extração de Dados (SQL)

### Qual é o volume de sessões, usuários únicos e transações por mês?

In [4]:
query_1 = """
SELECT
  DATE_TRUNC(PARSE_DATE('%Y%m%d', date), MONTH) as month_year,
  COUNT(DISTINCT visitId) AS unique_visits,
  COUNT(DISTINCT fullVisitorId) AS unique_users,
  SUM(totals.transactions) AS transactions
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  month_year
ORDER BY
  month_year
"""

unique_visits_df = client.query(query_1).to_dataframe()

display(unique_visits_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month_year,unique_visits,unique_users,transactions
0,2016-08-01,73494,61699,1241
1,2016-09-01,69729,59121,904
2,2016-10-01,95409,84901,919
3,2016-11-01,111201,99734,955
4,2016-12-01,77589,63839,1450
5,2017-01-01,63585,53041,713
6,2017-02-01,61122,51364,733
7,2017-03-01,68574,57888,993
8,2017-04-01,65935,55681,959
9,2017-05-01,64217,52233,1160


### Quais canais de aquisição (source/medium) geram mais receita? E quais têm melhor taxa de conversão?

In [12]:
query_2 = """
WITH revenue_per_channel AS (
  SELECT
    CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
    SUM(COALESCE(totals.totalTransactionRevenue, 0)) AS revenue
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  GROUP BY
    acquisition_channel
  ORDER BY
    revenue DESC
)

SELECT
  *,
  CONCAT(FORMAT('%.2f', (revenue / SUM(revenue) OVER ()) * 100), '%') AS revenue_share,
  CASE 
    WHEN acquisition_channel = '(direct) / (none)' THEN NULL 
    ELSE CONCAT(
           FORMAT(
             '%.2f', 
             (revenue / SUM(
                CASE WHEN acquisition_channel = '(direct) / (none)' THEN 0 ELSE revenue END
             ) OVER()) * 100
           ),
           '%'
         ) 
  END AS revenue_share_excluding_direct
FROM
  revenue_per_channel
ORDER BY
  revenue DESC
LIMIT 
  6
"""

source_revenue = client.query(query_2).to_dataframe()

display(source_revenue)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,revenue,revenue_share,revenue_share_excluding_direct
0,(direct) / (none),1332591980000,74.86%,None
1,google / organic,238974970000,13.42%,53.40%
2,dfa / cpm,128787150000,7.23%,28.78%
3,google / cpc,27876250000,1.57%,6.23%
4,mail.google.com / referral,24830780000,1.39%,5.55%
5,dealspotr.com / referral,5873640000,0.33%,1.31%


Acima, há o Top 6 canais que mais trazem receita para a empresa. Observa-se que o canal direto detém a grande maioria da receita. Excluindo-o para que se tenha uma noção melhor do share de receita entre os canais de marketing, observa-se que pesquisas no google orgânicas e "dfa/cpm" trazem quase 90% das demais receitas.

In [6]:
query_3 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f', 
      SUM(COALESCE(totals.transactions, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS conversion_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
ORDER BY
  SUM(COALESCE(totals.transactions, 0)) / COUNT(*) DESC
LIMIT 5
"""

convertion_rate_df = client.query(query_3).to_dataframe()

display(convertion_rate_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,conversion_rate
0,basecamp.com / referral,2,50.00%
1,mail.aol.com / referral,3,33.33%
2,us-mg5.mail.yahoo.com / referral,4,25.00%
3,calendar.google.com / referral,5,20.00%
4,chat.google.com / referral,7,14.29%


Observa-se, que o Top 5 de canais com a maior taxa de conversão é bem diferente dos canais com maior receita. Entretanto, isso se deve ao fato de terem um baixo número de sessões, logo, é mais fácil terem uma taxa de conversão maior. 

Ao se remover os canais com baixo número de sessões obtem-se a seguinte tabela:

In [7]:
query_4 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f', 
      SUM(COALESCE(totals.transactions, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS conversion_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
HAVING
  total_sessions > 100
ORDER BY
  SUM(COALESCE(totals.transactions, 0)) / COUNT(*) DESC
LIMIT 5
"""

convertion_rate_df_filtered = client.query(query_4).to_dataframe()

display(convertion_rate_df_filtered)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,conversion_rate
0,dealspotr.com / referral,528,7.58%
1,mail.google.com / referral,1457,4.80%
2,l.facebook.com / referral,795,3.77%
3,groups.google.com / referral,1025,3.71%
4,google / cpm,496,3.63%


Desta forma, conseguimos um dado mais confiável para qual é o canal com maior conversão. Neste caso, é o `dealspotr.com / referral` que também está presente nos top canais de receita, sendo o 6° colocado.

### Qual é a distribuição de sessões por dispositivo (desktop, mobile, tablet)?

In [8]:
query_5 = """
--sql
WITH sessions_per_device AS (
  SELECT
    device.deviceCategory AS device,
    COUNT(DISTINCT visitId) AS unique_sessions
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  GROUP BY
    device
)

SELECT
  *,
  CONCAT(FORMAT('%.2f', (unique_sessions / SUM(unique_sessions) OVER ()) * 100), '%') AS session_share
FROM
  sessions_per_device
ORDER BY
  unique_sessions DESC
"""

device_sessions_df = client.query(query_5).to_dataframe()

display(device_sessions_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,device,unique_sessions,session_share
0,desktop,653795,73.32%
1,mobile,207512,23.27%
2,tablet,30395,3.41%


Nota-se que a maioria das sessões vêm de usuários de Desktop, enquanto um número bem baixo delas vêm dos usuários de tablet.

### Quais são os 10 produtos mais vendidos em receita e em volume de unidades?

In [9]:
query_6 = """
WITH revenue_per_product AS (
  SELECT
    p.v2ProductName AS product_name,
    SUM(COALESCE(p.productRevenue, 0)) AS total_revenue
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_20170801`,
    UNNEST(hits) AS h,
    UNNEST(h.product) AS p
  GROUP BY
    product_name
)

SELECT
  *,
  CONCAT(FORMAT('%.2f', (total_revenue / SUM(total_revenue) OVER ()) * 100), '%') AS revenue_share
FROM
  revenue_per_product
ORDER BY
  total_revenue DESC
LIMIT
  10
"""

product_revenue_df = client.query(query_6).to_dataframe()

display(product_revenue_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_name,total_revenue,revenue_share
0,Google Leather Journal-Black,1215850000,13.66%
1,Google 17oz Stainless Steel Sport Bottle,1094480000,12.29%
2,Google Blackout Cap,749097272,8.41%
3,Google Power Bank,528056060,5.93%
4,26 oz Double Wall Insulated Bottle,463050475,5.20%
5,Google Spiral Journal with Pen,439770000,4.94%
6,Google Men's 100% Cotton Short Sleeve Hero Tee...,339627272,3.82%
7,Google Canvas Tote Natural/Navy,293843938,3.30%
8,Google Tri-blend Hoodie Grey,291970604,3.28%
9,Satin Black Ballpoint Pen,250000000,2.81%


In [11]:
query_7 = """
WITH quantity_per_product AS (
  SELECT
    p.v2ProductName AS product_name,
    SUM(COALESCE(p.productQuantity, 0)) AS product_quantity
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_20170801`,
    UNNEST(hits) AS h,
    UNNEST(h.product) AS p
  GROUP BY
    product_name
)

SELECT
  *,
  CONCAT(FORMAT('%.2f', (product_quantity / SUM(product_quantity) OVER ()) * 100), '%') AS product_quantity_share
FROM
  quantity_per_product
ORDER BY
  product_quantity DESC
LIMIT
  10
"""

product_quantity_df = client.query(query_7).to_dataframe()

display(product_quantity_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_name,product_quantity,product_quantity_share
0,Ballpoint LED Light Pen,455,9.33%
1,Google 17oz Stainless Steel Sport Bottle,334,6.85%
2,Leatherette Journal,319,6.54%
3,Google Spiral Journal with Pen,290,5.95%
4,Foam Can and Bottle Cooler,253,5.19%
5,Google Leather Journal-Black,250,5.13%
6,Google Sunglasses,228,4.68%
7,Google Blackout Cap,186,3.81%
8,Android 17oz Stainless Steel Sport Bottle,166,3.40%
9,Satin Black Ballpoint Pen,104,2.13%


Observa-se que os produtos mais vendidos são diferentes, em sua maioria, dos com maior receita. Indicando que o público tem preferência pelos produtos mais baratos da empresa, e que, apesar de serem vendidos em maior quantidade, não conseguem trazer o mesmo retorno de produtos mais caros que vendem menos.

### Qual é a taxa de rejeição (bounce rate) por canal? Existe diferença significativa entre eles?

In [15]:
query_8 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel, 
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f',
      SUM(COALESCE(totals.bounces, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS bounce_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
ORDER BY
  SUM(COALESCE(totals.bounces, 0)) / COUNT(*) DESC
"""

bounce_rate_df = client.query(query_8).to_dataframe()

display(bounce_rate_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,bounce_rate
0,takeout.google.com / referral,1,100.00%
1,images.google.lk / referral,1,100.00%
2,m.sp.sm.cn / referral,4,100.00%
3,it.pinterest.com / referral,1,100.00%
4,images.google.com.au / referral,1,100.00%
...,...,...,...
277,wanelo.com / referral,1,0.00%
278,google.be / referral,3,0.00%
279,images.google.pt / referral,1,0.00%
280,kr.pinterest.com / referral,3,0.00%


Como pode-se observar na tabela acima, há uma grande diferença do bounce rate de acordo com o canal. Mas, novamente, temos canais com pouquíssimas sessões. Logo, os removeremos para termos um número mais concreto.

In [16]:
query_9 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel, 
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f',
      SUM(COALESCE(totals.bounces, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS bounce_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
HAVING
  total_sessions > 100
ORDER BY
  SUM(COALESCE(totals.bounces, 0)) / COUNT(*) DESC
"""

bounce_rate_df_filtered = client.query(query_9).to_dataframe()

display(bounce_rate_df_filtered)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,bounce_rate
0,int.search.tb.ask.com / referral,110,79.09%
1,m.baidu.com / referral,142,71.83%
2,t.co / referral,1529,70.37%
3,baidu / organic,3356,67.79%
4,productforums.google.com / referral,364,67.58%
5,support.google.com / referral,101,67.33%
6,youtube.com / referral,212602,65.99%
7,l.facebook.com / referral,795,63.14%
8,m.facebook.com / referral,3365,61.04%
9,google.co.jp / referral,356,60.67%


Agora, mesmo com os dados filtrados, ainda chegamos na mesma conclusão: O canal é sim um fator que influencia o bounce rate.